# Local model check v2 (Kaggle, GPU T4 x2)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only**.

Same 8 smoke prompts as v1. This run covers:
1. The 3 new local models (granite4:350m, gemma3:1b, llama3.2:1b) + a re-test of qwen3:1.7b with the thinking fix.
2. CPU-only speed (2 threads) for those 4.
3. **Optional:** 3 bigger open models (12–14B) for the 'own GPU server' tier. Skip if short on GPU time.

Download `local_bench_v2_results.zip` from the Output panel at the end.

In [ ]:
%cd /kaggle/working
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

In [ ]:
%%writefile local_bench_ollama.py
"""
Run a list of Ollama models over a JSON prompt file on a free Kaggle/Colab T4.

Usage in a notebook cell:

    !python local_bench_ollama.py --prompts prompts.json --models qwen2.5:0.5b qwen3:1.7b \
        --out results_gpu.json
    # CPU-only latency pass (the deployment constraint: 2 threads):
    !python local_bench_ollama.py --prompts prompts.json --models qwen2.5:0.5b --cpu --out results_cpu.json

prompts.json: [{"id": "...", "category": "...", "prompt": "..."}, ...]
Output: one record per (model, prompt) with text, token counts, tokens/s, wall latency.
Models are pulled one at a time and removed after use to keep the disk small.
Note: Kaggle/Colab CPUs differ from the laptop, so CPU latency measured there is
indicative only; re-measure the final candidates on the laptop.
"""

import argparse
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

HOST = "http://127.0.0.1:11434"


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


def install_and_serve() -> None:
    if not shutil.which("ollama"):
        # zstd is needed by the Ollama install script on some images
        sh("apt-get update -qq && apt-get install -y -qq zstd >/dev/null 2>&1 || true")
        sh("curl -fsSL https://ollama.com/install.sh | sh")
    try:
        urllib.request.urlopen(f"{HOST}/api/version", timeout=2)
        return
    except Exception:  # noqa: BLE001
        pass
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(f"{HOST}/api/version", timeout=2)
            return
        except Exception:  # noqa: BLE001
            time.sleep(1)
    sys.exit("ollama server did not start; see /tmp/ollama.log")


def generate(model: str, prompt: str, options: dict) -> dict:
    # keep_alive 10m: the model stays loaded for the rest of its prompts (keep_alive 0
    # reloaded it on every prompt, so latency would mostly measure load time).
    # think False: qwen3 otherwise spends hundreds of hidden tokens per answer.
    payload = {"model": model, "prompt": prompt, "stream": False,
               "keep_alive": "10m", "options": options}
    if model.startswith("qwen3"):  # only send `think` to thinking-capable models
        payload["think"] = False
    body = json.dumps(payload).encode()
    req = urllib.request.Request(f"{HOST}/api/generate", body, {"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=600) as r:
        d = json.load(r)
    ms = int((time.perf_counter() - t0) * 1000)
    ec, ed = d.get("eval_count", 0), d.get("eval_duration", 0)
    raw = d.get("response", "")
    # Thinking models can still emit <think>...</think> (or a stray /no_think) even with
    # think=False. Keep the raw text, but grade only the visible answer.
    text = re.sub(r"<think>.*?(</think>|$)", "", raw, flags=re.S)
    text = text.replace("/no_think", "").strip()
    return {"text": text, "raw_text": raw if raw != text else None,
            "thinking_leaked": raw != text, "prompt_tokens": d.get("prompt_eval_count"),
            "completion_tokens": ec, "tokens_per_s": round(ec / (ed / 1e9), 1) if ed else None,
            "load_ms": int(d.get("load_duration", 0) / 1e6), "latency_ms": ms}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompts", required=True)
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--out", default="results.json")
    ap.add_argument("--cpu", action="store_true", help="num_gpu=0, num_thread=2")
    ap.add_argument("--max-tokens", type=int, default=512)
    ap.add_argument("--keep-models", action="store_true")
    a = ap.parse_args()

    prompts = json.load(open(a.prompts, encoding="utf-8"))
    opts = {"temperature": 0, "num_predict": a.max_tokens}
    if a.cpu:
        opts.update(num_gpu=0, num_thread=2)

    install_and_serve()
    results = json.load(open(a.out)) if os.path.exists(a.out) else []
    done = {(r["model"], r["id"]) for r in results}
    for model in a.models:
        sh(f"ollama pull {model}")
        for p in prompts:
            if (model, p["id"]) in done:
                continue
            try:
                rec = generate(model, p["prompt"], opts)
            except Exception as e:  # noqa: BLE001
                rec = {"error": str(e)[:300]}
            results.append({"model": model, "id": p["id"], "category": p.get("category"),
                            "mode": "cpu_2threads" if a.cpu else "gpu", **rec})
            json.dump(results, open(a.out, "w", encoding="utf-8"), indent=1)  # resumable
        sh(f"ollama stop {model} || true")  # free GPU/RAM before the next model
        if not a.keep_models:
            sh(f"ollama rm {model}")
    print(f"wrote {len(results)} records to {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prompts.json
[
 {
  "id": "smoke-fact",
  "category": "factual",
  "prompt": "Explain in two sentences what a hash table is and why lookups are fast."
 },
 {
  "id": "smoke-math",
  "category": "math",
  "prompt": "A shop has 480 notebooks. It sells 25% on Monday and 90 more on Tuesday. How many notebooks are left? Answer with the number."
 },
 {
  "id": "smoke-sent",
  "category": "sentiment",
  "prompt": "Classify the sentiment of this review as positive, negative or mixed, then give a one-line reason: 'Delivery was quick, but the headphones stopped charging after two days.'"
 },
 {
  "id": "smoke-summ",
  "category": "summarisation",
  "prompt": "Summarise in exactly one sentence: The city council approved a new bike-lane network after a year of public consultations. The plan adds 40 km of protected lanes over three years, funded partly by a national grant. Local businesses raised concerns about losing parking, so the council promised a review after the first phase."
 },
 {
  "id": "smoke-ner",
  "category": "ner",
  "prompt": "Extract all named entities with their types (PERSON, ORG, LOCATION, DATE) as JSON from: 'Priya Nair joined Infosys in Pune on 3 March 2021.'"
 },
 {
  "id": "smoke-debug",
  "category": "code_debug",
  "prompt": "This function should return the average of a list but has a bug. Find and fix it:\ndef avg(xs):\n    return sum(xs) / len(xs) - 1"
 },
 {
  "id": "smoke-logic",
  "category": "logic",
  "prompt": "Ana, Ben and Cy each own a different pet: cat, dog, fish. Ben does not own the fish. Cy owns the dog. Who owns the cat?"
 },
 {
  "id": "smoke-codegen",
  "category": "code_gen",
  "prompt": "Write a Python function `second_largest(nums)` that returns the second-largest distinct number in a list, or None if it does not exist."
 }
]

## Step 1: GPU, new models + qwen3 re-test (~15–25 min)

In [ ]:
!python local_bench_ollama.py --prompts prompts.json --models granite4:350m gemma3:1b llama3.2:1b qwen3:1.7b --out results_v2_gpu.json --keep-models

## Step 2: CPU only, 2 threads (~15–30 min)
Models are already downloaded from step 1.

In [ ]:
!python local_bench_ollama.py --prompts prompts.json --models granite4:350m gemma3:1b llama3.2:1b qwen3:1.7b --cpu --out results_v2_cpu.json

## Step 3 (optional): bigger models for the 'own GPU server' tier (~30–45 min, ~25 GB of downloads)
Each is pulled, run on GPU, then deleted to free disk.

In [ ]:
!python local_bench_ollama.py --prompts prompts.json --models gemma3:12b phi4:14b qwen2.5:14b --out results_v2_gpu_large.json

## Step 4: Summary + download

In [ ]:
import json, pandas as pd
rows=[]
for f in ['results_v2_gpu.json','results_v2_cpu.json','results_v2_gpu_large.json']:
    try: rows += json.load(open(f))
    except FileNotFoundError: pass
df=pd.DataFrame(rows)
print('errors:', df['error'].notna().sum() if 'error' in df else 0)
cols=[c for c in ['tokens_per_s','latency_ms','completion_tokens'] if c in df]
display(df.groupby(['mode','model'])[cols].median().round(1))
if 'thinking_leaked' in df: display(df.groupby('model')['thinking_leaked'].sum())
pd.set_option('display.max_colwidth',120)
display(df[df['mode']=='gpu'][['model','category','text']])

In [ ]:
!zip -q local_bench_v2_results.zip results_v2_*.json prompts.json && ls -lh local_bench_v2_results.zip